# 01 - Train all 8 classes: how much of our own (Mae Suai) data do we need?
- **Train on:** all OpenEarthMap (OEM) images + **0% / 25% / 50% / 100%** of our Mae Suai images (these are the **seen** images).
- **Test on:** the Mae Suai images **not** used for training (the **unseen** share): 0% -> 100% unseen, 25% -> 75% unseen, 50% -> 50% unseen. At 100% nothing is unseen, so the score is measured on the training images (fit only, not generalization).
- Image and mask are matched by **same file name** (`1.tif` image <-> `1.tif` mask in `own_paired`; notebook 01 section 2 makes this folder).
- **8 classes:** bareland, rangeland, developed_space, road, tree, water, agriculture, building (mask values 1 to 8).
- All settings (classes, model, training) come from `configs/land_cover_8class.yaml`. To switch model, change `model.active` there.
- The code lives in `src/` (data, models, training, plots). This notebook only sets the options, runs the training and draws the results.

## 1. Setup

In [10]:
%load_ext autoreload
%autoreload 2

import os, sys, json
import torch
import pandas as pd
REPO_DIR = r"C:\land_cover_senior2026"
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

from src.train import load_config
from src.data.paths import resolve_paths
from src.data.pairing import pair_folder, shuffle_pairs, split_for
from src.data.seg_dataset import get_oem_ids
from src.experiments.oem_fraction import run_fraction, run_kfold, summarize_kfold
from src.visualization.plots import (print_table, plot_miou, plot_confusions, plot_iou_heatmap,
                                     plot_prediction_by_filename, print_metrics_table, setup_viewer)


cfg = load_config("configs/oem_8class.yaml")

# ---- CHOOSE THE MODEL HERE (no need to edit the yaml) ----
MODEL_CHOICE = "fpn_r34"       # unet_r34 | fpn_r34 | fpn_swin_b
assert MODEL_CHOICE in cfg["models"], f"pick one of {list(cfg['models'])}"
cfg["model"]["active"] = MODEL_CHOICE

# ---- the options you can change ----
FRACTIONS = [0.0, 0.25, 0.5, 0.75, 1.0]         # share of each fold's training pool used (seen) in training
K = cfg["train"].get("kfold", 5)                # number of rotating test folds (set `kfold:` under `train:` in the yaml)

P = resolve_paths(cfg)
CLASSES    = cfg["classes"]
MODEL_NAME = cfg["model"]["active"]
SEED       = cfg["train"]["seed"]
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "| model:", MODEL_NAME, "|", cfg["models"][MODEL_NAME], "| classes:", CLASSES, "| folds:", K)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Device: cuda | model: fpn_r34 | {'decoder': 'fpn', 'encoder': 'resnet34', 'encoder_weights': 'imagenet'} | classes: ['bareland', 'rangeland', 'developed_space', 'road', 'tree', 'water', 'agriculture', 'building'] | folds: 5


## 2. Pair images with masks and make the fixed split
Images and masks are matched by the same file name. All Mae Suai images are shuffled once (seed in the yaml) and saved to `results/own_split.json`.
For a fraction `f`, the first `f` of the shuffled list is **seen** (training) and the rest is **unseen** (test). So the 25% images are always inside the 50% images, and those inside the 100% images.

In [11]:
from src.data.pairing import pair_images_and_labels

own_pairs = pair_folder(P["own_images"], P["own_masks"])
oem_pairs = pair_images_and_labels(P["oem_root"])  # scans all OEM regions automatically
print("Mae Suai pairs:", len(own_pairs), "| OEM pairs:", len(oem_pairs))
assert len(own_pairs) > 0, "No Mae Suai pairs - check own_images and own_masks in the yaml"

# preview of the rotating folds (run_kfold builds the real ones when training starts)
n = len(own_pairs)
sizes = [n // K + (1 if i < n % K else 0) for i in range(K)]
print(f"Mae Suai total {n} | {K} folds | test sizes {sizes}")
for i, t in enumerate(sizes):
    pool = n - t
    counts = [round(f * pool) for f in FRACTIONS]
    print(f"  fold {i + 1}: test {t} unseen | pool {pool} | train on {counts} of {pool} at "
          f"{[int(f * 100) for f in FRACTIONS]}%")

OWN_IDS = cfg["mask_ids"]["own"]
OEM_IDS, oem_seen = get_oem_ids(cfg, oem_pairs)
print("own mask ids:", OWN_IDS)
print("oem mask ids:", OEM_IDS, "| values seen in OEM labels:", sorted(oem_seen) if oem_seen else "n/a")

Successfully paired 2687 images with their matching labels across all OEM regions.
Mae Suai pairs: 44 | OEM pairs: 2687
Mae Suai total 44 | 5 folds | test sizes [9, 9, 9, 9, 8]
  fold 1: test 9 unseen | pool 35 | train on [0, 9, 18, 26, 35] of 35 at [0, 25, 50, 75, 100]%
  fold 2: test 9 unseen | pool 35 | train on [0, 9, 18, 26, 35] of 35 at [0, 25, 50, 75, 100]%
  fold 3: test 9 unseen | pool 35 | train on [0, 9, 18, 26, 35] of 35 at [0, 25, 50, 75, 100]%
  fold 4: test 9 unseen | pool 35 | train on [0, 9, 18, 26, 35] of 35 at [0, 25, 50, 75, 100]%
  fold 5: test 8 unseen | pool 36 | train on [0, 9, 18, 27, 36] of 36 at [0, 25, 50, 75, 100]%
own mask ids: {'bareland': [1], 'rangeland': [2], 'developed_space': [3], 'road': [4], 'tree': [5], 'water': [6], 'agriculture': [7], 'building': [8], 'ignore': []}
oem mask ids: {'auto': False, 'bareland': [1], 'rangeland': [2], 'developed_space': [3], 'road': [4], 'tree': [5], 'water': [6], 'agriculture': [7], 'building': [8], 'ignore': []} | v

## 3. Train (one model per option)
This trains one model per option, so it takes a while. Change `FRACTIONS` in section 1 to run fewer.
Same recipe as `train.py` (cross-entropy + Dice, Adam, mixed precision). The score reported is from the **last epoch**, so we do not pick the best epoch using the test images.

In [ ]:
all_results = run_kfold(FRACTIONS, cfg, device, oem_pairs, own_pairs, OEM_IDS, OWN_IDS,
                        P["weights_dir"], P["results_dir"], MODEL_NAME, k=K, seed=SEED)


##### FOLD 1/5 | pool 35 | test 9 #####

=== fpn_r34 (Encoder: resnet34 | Decoder: fpn) | 0% | fold 0 ===
Images trained on: 2687 OEM + 0 Mae Suai = 2687 unique images
Samples per epoch (Mae Suai x5 repeat): 2687
Test on 9 held-out Mae Suai images


## 4. Results (each option is tested on the Mae Suai images it did not see; 100% is tested on its own training images)

In [ ]:
print_table(results, CLASSES)
plot_miou(results, save_to=P["results_dir"] / f"own_fraction_{MODEL_NAME}_miou.png")

## 5. Heatmaps: where did it go wrong?
- **Confusion matrices** (one per option): each row is the true class, each column is what the model predicted. Numbers are the % of that true class, so the diagonal is correct and every off-diagonal cell is a specific mistake (e.g. true *road* predicted as *developed_space*).
- **IoU + mIoU heatmap**: one row per option, one column per class and the overall mIoU, for a side-by-side comparison.

In [ ]:
import numpy as np

KEY = "miou"   # same key name you use in summarize_kfold

# score each fold by its average mIoU across all fractions
fold_scores = {k: np.mean([r[KEY] for r in res]) for k, res in all_results.items()}
best_fold = max(fold_scores, key=fold_scores.get)
print("fold scores:", {k: round(v, 3) for k, v in fold_scores.items()})
print("best fold:", best_fold)

results = all_results[best_fold]

plot_confusions(results, CLASSES, save_to=P["results_dir"] / f"own_fraction_{MODEL_NAME}_confusion_bestfold{best_fold}.png")
plot_iou_heatmap(results, CLASSES, save_to=P["results_dir"] / f"own_fraction_{MODEL_NAME}_iou_heatmap_bestfold{best_fold}.png")

In [ ]:
from src.visualization.plots import print_metrics_table

dfs = []
for k, fold_results in all_results.items():
    print(f"\n--- fold {k} ---")
    d = print_metrics_table(fold_results, CLASSES,
                            save_to=P["results_dir"] / f"own_fraction_{MODEL_NAME}_metrics_fold{k}.csv")
    d = d.copy()
    d.insert(0, "fold", k)
    dfs.append(d)

# one combined table with every fold
df_all = pd.concat(dfs, ignore_index=True)
df_all.to_csv(P["results_dir"] / f"own_fraction_{MODEL_NAME}_metrics_all_folds.csv", index=False)
df_all

In [ ]:
from src.visualization.plots import setup_viewer
show = setup_viewer(cfg, own_pairs, OWN_IDS, P, device)
show("85.tif")

## Notes
- Each option has a **different test set** (0% tests on all images, 25% on 75%, 50% on 50%), so the scores are not measured on identical images. Small differences can be noise.
- The 100% option has no unseen images, so its score only shows how well the model fits its training data. Do not read it as generalization.
- The split order is listed in `results/own_split.json`. Weights and results files include the model name, so different models do not overwrite each other.
- Before committing to GitHub: *Kernel > Restart & Clear Output*.
- This notebook uses 8 classes, so the confusion matrices are 8x8. Small classes (e.g. road, bareland) usually get a lower IoU than big ones (e.g. tree, rangeland).